# 从零实现注意力机制（教学版）

本 notebook 基于《Build a Large Language Model From Scratch》第3章内容，通过详细的代码和讲解，帮助你理解：
- 为什么需要注意力机制
- 自注意力（无权重 / 可训练权重）
- 因果注意力（掩码）
- 多头注意力

**环境要求**：PyTorch 2.0+

In [1]:
from importlib.metadata import version
print("torch version:", version("torch"))
import torch
import torch.nn as nn

torch version: 2.5.1


## 1. 为什么需要注意力机制？

传统 RNN 将整个输入序列压缩成一个固定长度的向量，导致长序列信息丢失。

**注意力机制**允许模型在生成每个输出时，动态地“关注”输入序列中不同位置的信息，并根据相关性分配权重。

**自注意力**则是在同一序列内部计算每个元素与其他所有元素的关联，生成更丰富的上下文表示。

## 2. 最简单的自注意力（无训练权重）

我们先理解核心计算流程：**加权求和**。

输入：6 个词，每个词用 3 维向量表示（实际应用中维度会更大）。

In [3]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your
   [0.55, 0.87, 0.66], # journey
   [0.57, 0.85, 0.64], # starts
   [0.22, 0.58, 0.33], # with
   [0.77, 0.25, 0.10], # one
   [0.05, 0.80, 0.55]] # step
)
print("输入形状:", inputs.shape)  # (6, 3)

输入形状: torch.Size([6, 3])


### 2.1 以第2个词（journey）为查询，计算注意力分数

注意力分数 = 查询向量与所有输入向量的点积，代表相似度。

In [4]:
query = inputs[1]   # journey
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print("未归一化的注意力分数:", attn_scores_2)

未归一化的注意力分数: tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


### 2.2 归一化得到注意力权重（使用 softmax）

softmax 使所有权重为正且和为 1，便于解释为概率。

In [6]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("注意力权重:", attn_weights_2)
print("权重和:", attn_weights_2.sum().item())

注意力权重: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
权重和: 1.0


### 2.3 计算上下文向量

上下文向量 = 权重 × 输入向量的加权和。

In [7]:
context_vec_2 = torch.zeros(query.shape)
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i
print("第2个词的上下文向量:", context_vec_2)

第2个词的上下文向量: tensor([0.4419, 0.6515, 0.5683])


### 2.4 为所有 token 并行计算（矩阵乘法）

上述循环效率低，改用矩阵运算：先计算所有注意力分数矩阵，再 softmax，最后乘以输入矩阵。

In [13]:
attn_scores = inputs @ inputs.T   # 6x6 矩阵，每个元素是点积
print("点积:\n", attn_scores)
attn_weights = torch.softmax(attn_scores, dim=-1)  # 按行归一化
print("按行归一化:\n", attn_weights)

all_context_vecs = attn_weights @ inputs   # 6x3
print("所有上下文向量:\n", all_context_vecs)
print("形状:", all_context_vecs.shape)

点积:
 tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
按行归一化:
 tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])
所有上下文向量:
 tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
形状: torch.Size([6, 3])


## 3. 带可训练权重的自注意力（Scaled Dot-Product Attention）

真实 Transformer 中使用**可训练矩阵**将输入映射为 **查询（Query）**、**键（Key）**、**值（Value）**。

公式：
$$
\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V
$$

其中 $d_k$ 是键向量的维度。

In [21]:
d_in = inputs.shape[1]   # 输入维度 = 3
d_out = 2                # 输出维度（可任意）

torch.manual_seed(123)
W_query = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

# 验证：将输入投影到 2 维空间
queries = inputs @ W_query
keys    = inputs @ W_key
values  = inputs @ W_value
print("queries 形状:", queries.shape)  # (6,2)

queries 形状: torch.Size([6, 2])


### 3.1 计算缩放点积注意力（以第2个词为例）

In [ ]:
query_2 = queries[1]
attn_scores_2 = query_2 @ keys.T   # 点积

d_k = keys.shape[-1]   # =2
attn_weights_2 = torch.softmax(attn_scores_2 / (d_k ** 0.5), dim=-1)
context_vec_2 = attn_weights_2 @ values

print("注意力权重:", attn_weights_2)
print("上下文向量:", context_vec_2)

### 3.2 封装为 `SelfAttention` 类

使用 `nn.Linear` 代替手动参数，因为 `nn.Linear` 有更好的权重初始化。

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys    = self.W_key(x)
        values  = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / (keys.shape[-1]**0.5), dim=-1)
        return attn_weights @ values

torch.manual_seed(123)
sa = SelfAttention(d_in, d_out)
out = sa(inputs)
print("输出形状:", out.shape)   # (6,2)

## 4. 因果注意力（Causal Attention）

在自回归生成中，模型不能看到未来 token。我们通过在 softmax **之前**将未来位置的注意力分数设为 `-inf` 来实现掩码。

### 4.1 构造上三角掩码

In [ ]:
context_length = inputs.shape[0]
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
print("掩码矩阵（1 表示需要掩盖的位置）:\n", mask)

### 4.2 对注意力分数应用掩码

In [ ]:
# 使用之前的 SelfAttention 类获得分数
queries = sa.W_query(inputs)
keys    = sa.W_key(inputs)
attn_scores = queries @ keys.T

masked_scores = attn_scores.masked_fill(mask.bool(), -torch.inf)
attn_weights_causal = torch.softmax(masked_scores / (keys.shape[-1]**0.5), dim=-1)

print("因果注意力权重（每行和为1，未来位置为0）:\n", attn_weights_causal)

### 4.3 添加 Dropout（训练时正则化）

Dropout 随机将部分注意力权重置零，并缩放剩余值。

In [ ]:
torch.manual_seed(123)
dropout = nn.Dropout(0.5)
attn_weights_dropped = dropout(attn_weights_causal)
print("经过 Dropout 后的权重（部分被置零）:\n", attn_weights_dropped)

### 4.4 完整的因果注意力类（支持 batch）

注意：为了处理 batch 输入，我们需要对 `keys.transpose(1,2)` 进行正确的维度操作。

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x)  # (b, num_tokens, d_out)
        keys    = self.W_key(x)
        values  = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)  # (b, num_tokens, num_tokens)
        mask = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask, -torch.inf)

        attn_weights = torch.softmax(attn_scores / (keys.shape[-1]**0.5), dim=-1)
        attn_weights = self.dropout(attn_weights)
        return attn_weights @ values

# 测试：构造一个 batch（两个相同的输入）
batch = torch.stack((inputs, inputs), dim=0)  # (2,6,3)
ca = CausalAttention(d_in=3, d_out=2, context_length=6, dropout=0.0)
out_batch = ca(batch)
print("batch 输出形状:", out_batch.shape)  # (2,6,2)

## 5. 多头注意力（Multi-Head Attention）

多头注意力并行运行多个注意力头，每个头学习不同的特征子空间，最后拼接结果。

### 5.1 简单实现：包装多个因果注意力层

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList([
            CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
            for _ in range(num_heads)
        ])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

mha_wrapper = MultiHeadAttentionWrapper(d_in=3, d_out=2, context_length=6, dropout=0.0, num_heads=2)
out_wrapper = mha_wrapper(batch)
print("多头（包装）输出形状:", out_wrapper.shape)   # (2,6,4)  因为 2个头 * 2维 = 4

### 5.2 高效实现：单矩阵切分

实际生产中，我们会使用一个大的权重矩阵，然后通过 `view` 和 `transpose` 拆分为多个头，这样计算效率更高。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # 最终投影
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # 线性变换并拆分为多个头
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim)
        keys    = self.W_key(x).view(b, num_tokens, self.num_heads, self.head_dim)
        values  = self.W_value(x).view(b, num_tokens, self.num_heads, self.head_dim)

        # 转置以便在头之间独立计算注意力
        queries = queries.transpose(1, 2)  # (b, num_heads, num_tokens, head_dim)
        keys    = keys.transpose(1, 2)
        values  = values.transpose(1, 2)

        # 计算注意力分数并应用掩码
        attn_scores = queries @ keys.transpose(-2, -1)  # (b, num_heads, num_tokens, num_tokens)
        mask = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask, -torch.inf)

        attn_weights = torch.softmax(attn_scores / (keys.shape[-1]**0.5), dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 加权求和并重组
        context = (attn_weights @ values).transpose(1, 2)  # (b, num_tokens, num_heads, head_dim)
        context = context.contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context)

mha_efficient = MultiHeadAttention(d_in=3, d_out=4, context_length=6, dropout=0.0, num_heads=2)
out_efficient = mha_efficient(batch)
print("高效多头注意力输出形状:", out_efficient.shape)  # (2,6,4)

## 6. 总结

- **自注意力**：通过加权求和将每个 token 与序列中所有 token 关联。
- **可训练权重**：通过 W_q, W_k, W_v 将输入映射到不同空间。
- **缩放点积**：除以 √d_k 防止梯度消失。
- **因果注意力**：使用上三角掩码禁止未来信息。
- **Dropout**：在注意力权重上应用，防止过拟合。
- **多头注意力**：并行多个头，捕捉不同子空间特征。

以上代码是构建 GPT 等大语言模型中注意力模块的核心。